# **Radio Events Extraction Full Pipeline**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()      # Hides standard warnings
hf_logging.disable_progress_bar()     # Hides the red adapter loading bars

# Install required Libararies

In [ ]:
!pip install "numpy<2.0" transformers accelerate bitsandbytes groq

In [ ]:
!pip install librosa soundfile torchaudio pydub openai-whisper faster-whisper pyannote.audio

In [ ]:
!pip install pycountry sentencepiece gliner dateparser regex symspellpy deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 155.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 21.0 MB/s eta 0:00:00


# Radio Ingestion & Filtering

In [ ]:
import requests
import pandas as pd
import time

pd.set_option('display.max_colwidth', None)

def fetch_conflict_news_stations():
    # Looking for ALL stations in target countries
    targets = [
        {"countrycode": "ET"},
        {"countrycode": "UG"},
        {"countrycode": "CD"}
    ]

    headers = {'User-Agent': 'ACLED_Audio_News_Extractor/1.0'}
    base_url = "https://de1.api.radio-browser.info/json/stations/search"

    all_stations = []

    print("Querying Radio-Browser API (Broad Net)...\n")

    for target in targets:
        params = {
            "hidebroken": "true",
            "reverse": "true",
            "tag": "news"
        }
        params.update(target)

        try:
            response = requests.get(base_url, headers=headers, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            for station in data:
                all_stations.append({
                    "Name": station.get("name", "Unknown").strip(),
                    "Country": station.get("countrycode", ""),
                    "Language": station.get("language", ""),
                    "Tags": station.get("tags", ""),
                    "Direct_Stream_URL": station.get("url_resolved", "")
                })

            print(f"Pulled {len(data)} stations for: {target}")
            time.sleep(1)

        except requests.exceptions.RequestException as e:
            print(f"Failed to fetch {target}: {e}")

    df = pd.DataFrame(all_stations)
    return df

# 1. Get the massive unfiltered list
raw_df = fetch_conflict_news_stations()

# 2. Filter locally using Pandas
# Look for stations that might be news based on their name or tags
search_terms = ['news', 'radio', 'fm', 'actualite', 'zena', 'info', 'talk', 'current afairs']
pattern = '|'.join(search_terms)

# Filter the dataframe to only keep rows where the Name OR the Tags contain our keywords
news_likely_df = raw_df[
    raw_df['Name'].str.contains(pattern, case=False, na=False) |
    raw_df['Tags'].str.contains(pattern, case=False, na=False)
]

print("\nFiltered Results:")
display(news_likely_df)

Querying Radio-Browser API (Broad Net)...

Pulled 1 stations for: {'countrycode': 'ET'}
Pulled 30 stations for: {'countrycode': 'UG'}
Pulled 2 stations for: {'countrycode': 'CD'}

Filtered Results:


,Name,Country,Language,Tags,Direct_Stream_URL
0,DW Amharic,ET,oromo amharic somali,news,https://dw.audiostream.io/dw/1027/mp3/64/dw08
1,Times Radio UK,UG,english,🌍news🌏,http://timesradio.wireless.radio/stream
2,Times Radio,UG,english,current affairs|news,http://timesradio.wireless.radio/stream
3,TALKSPORT,UG,english,sports news,http://radio.talksport.com/stream2?awparams=platform:ts-web&amsparams=playerid:ts-web;
4,TALKRADIO,UG,english,news,https://radio.talkradio.co.uk/stream
5,talkRADIO,UG,english,news and current affairs,https://radio.talkradio.co.uk/stream
6,Skynews Radio,UG,english,news,https://video.news.sky.com/snr/news/snrnews.mp3
7,SKY NEWS,UG,english,news,https://video.news.sky.com/snr/news/snrnews.mp3
8,Rock FM Uganda - Kampala (MP3),UG,"english,luganda","entertainment,music,news",http://titan.shoutca.st:8341/;
9,Radio Yoo (MP3),UG,"english,luganda","christian,entertainment,gospel,hits,news,religious,talk",http://stream.zeno.fm/v73tc5gwaphvv


# VAD Identification

In [ ]:
import os
import time
import requests
import torch

# 1. INITIALIZE VAD MODEL
print("Loading Silero VAD model...")
vad_model, vad_utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                              model='silero_vad',
                              force_reload=False)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = vad_utils


# 2. THE RECORDER
def record_live_stream(stream_url, output_filename, duration_seconds=30):
    print(f" Attempting to record from: {stream_url}")

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36'
    }

    try:
        response = requests.get(stream_url, headers=headers, stream=True, timeout=10)
        response.raise_for_status()

        start_time = time.time()
        with open(output_filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if time.time() - start_time > duration_seconds:
                    break
                if chunk:
                    f.write(chunk)

        print(f"Audio saved to: {output_filename}")
        return True

    except requests.exceptions.RequestException as e:
        print(f"Connection failed: {e}")
        return False


# 3. THE VAD EVALUATOR
def evaluate_audio_content(filepath, speech_threshold=0.3):
    print(f"Analyzing audio for human speech...")

    # 1. Setup the device (GPU if available, otherwise CPU)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Ensure the model is on the correct device
    vad_model.to(device)

    try:
        # Read the audio into CPU memory
        wav = read_audio(filepath, sampling_rate=16000)

        # Move the audio tensor to the exact same device as the model
        wav = wav.to(device)

        # Run the model
        speech_timestamps = get_speech_timestamps(wav, vad_model, sampling_rate=16000)

        # Calculate speech ratio
        total_frames = len(wav)
        speech_frames = sum([segment['end'] - segment['start'] for segment in speech_timestamps])
        speech_ratio = speech_frames / total_frames if total_frames > 0 else 0

        print(f"Speech detected: {speech_ratio:.1%}")

        if speech_ratio >= speech_threshold:
            print("Decision: KEEP (Human speech detected!)")
            return True
        else:
            print("Decision: DELETE (Music/Dead air detected.)")
            os.remove(filepath)
            return False

    except Exception as e:
        print(f"Error analyzing audio: {e}")
        if os.path.exists(filepath):
            os.remove(filepath)
        return False

# 4. THE PIPELINE BRIDGE
import subprocess

def wait_for_speech_and_sample(stream_url, output_filename="sample.mp3", duration=30):
    """
    Records a chunk, cleans the audio format, tests for speech,
    and returns the clean filepath ONLY if speech is found.
    """
    # 1. Record the raw chunk
    success = record_live_stream(stream_url, output_filename, duration_seconds=duration)

    if not success:
        return None

    # 2. Clean the audio using FFmpeg
    # This prevents Torchaudio from crashing on broken MP3 frames
    clean_wav = "clean_sample.mp3"
    try:
        subprocess.run(['ffmpeg', '-y', '-i', output_filename, '-ar', '16000', '-ac', '1', clean_wav],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
    except Exception as e:
        print(f"Audio cleaning failed: {e}")
        return None

    # 3. Evaluate the CLEAN file
    has_speech = evaluate_audio_content(clean_wav, speech_threshold=0.3)

    # 4. Return result
    if has_speech:
        return clean_wav # pass the clean WAV to stt models
    else:
        return None

Loading Silero VAD model...
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip


# Diarization

In [ ]:
import torch
from pyannote.audio import Pipeline
from google.colab import userdata

hf_token = userdata.get(' Add Your HF TOKEN Here')

print("Loading PyAnnote Diarization Model (Community-1)...")

# Requesting the native 4.0 model and passing the token
diarization_model = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=hf_token
)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diarization_model.to(device)
print("Diarization Model successfully loaded into GPU")

Loading PyAnnote Diarization Model (Community-1)...
Diarization Model successfully loaded into GPU


In [ ]:
def run_diarization(audio_path):
    """Analyzes the audio and returns a raw list of speaker timestamps."""
    print(f"Running Speaker Diarization on {audio_path}...")
    try:
        output = diarization_model(audio_path)
        diarization = output.speaker_diarization

        speaker_segments = []

        for turn, _, speaker in diarization.itertracks(yield_label=True):
            segment = {
                "start": turn.start,
                "end": turn.end,
                "speaker": speaker
            }
            speaker_segments.append(segment)

        # Return the raw data list so the next function can use it
        return speaker_segments

    except Exception as e:
        print(f"Diarization failed: {e}")
        return []

In [ ]:
def merge_speaker_segments(segments, max_gap=2.0):
    """
    Glues together back-to-back segments from the same speaker
    if the silence between them is short (e.g., taking a breath).
    """
    if not segments:
        return []

    merged = [segments[0]]
    for current in segments[1:]:
        previous = merged[-1]

        # If it's the same speaker AND the pause is less than 'max_gap' seconds
        if current['speaker'] == previous['speaker'] and (current['start'] - previous['end']) <= max_gap:
            # Merge them by extending the end time!
            previous['end'] = current['end']
        else:
            merged.append(current)

    return merged

In [ ]:
import os
import subprocess

def transcribe_and_stitch_diarization(audio_path, raw_speaker_segments, target_language):
    print(f"\nStarting Smart Diarization Stitching for: {audio_path}")
    print(f"   -> Target Language: {target_language}")

    # THE FIX: Glue the stuttering segments together!
    speaker_segments = merge_speaker_segments(raw_speaker_segments, max_gap=2.0)
    print(f"   -> Smoothed {len(raw_speaker_segments)} raw voice blips into {len(speaker_segments)} continuous blocks.")

    final_script = []
    is_first_chunk = True

    for i, segment in enumerate(speaker_segments):
        start_time = segment["start"]
        end_time = segment["end"]
        speaker = segment["speaker"]

        # THE FIX: Increased from 0.5 to 1.5 seconds. Ignore random sighs and coughs!
        if (end_time - start_time) < 1.5:
            continue

        temp_chunk_name = f"temp_speaker_chunk_{i}.wav"

        try:
            subprocess.run([
                'ffmpeg', '-y', '-i', audio_path,
                '-ss', str(start_time), '-to', str(end_time),
                '-c:a', 'pcm_s16le', '-ar', '16000', '-ac', '1',
                temp_chunk_name
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except Exception as e:
            continue

        chunk_text = targeted_chunk_router(temp_chunk_name, target_language, verbose=is_first_chunk)

        if is_first_chunk:
            print(f"   -> Model loaded successfully. Processing remaining chunks quietly...")
            is_first_chunk = False

        if chunk_text and isinstance(chunk_text, str) and chunk_text.strip():
            script_line = f"[{speaker}]: {chunk_text.strip()}"
            final_script.append(script_line)

        if os.path.exists(temp_chunk_name):
            os.remove(temp_chunk_name)

    final_stitched_transcript = "\n".join(final_script)
    print("Script stitching complete!")
    return final_stitched_transcript

# Language Detector

In [ ]:
import torch
import librosa
from transformers import Wav2Vec2ForSequenceClassification, AutoFeatureExtractor
import pycountry

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device.upper()}")

# LOAD LANGUAGE DETECTOR
print(" Loading Language Detector...")
identifier_model_id = "facebook/mms-lid-1024"

processor = AutoFeatureExtractor.from_pretrained(identifier_model_id)
detect_model = Wav2Vec2ForSequenceClassification.from_pretrained(identifier_model_id)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
detect_model.to(device)

Using Device: CUDA
 Loading Language Detector...


Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=

In [ ]:
def detect_language_mms(audio_path):
    print(f"Listening to {audio_path}...")

    # Load audio and listen for 30s
    audio, sr = librosa.load(audio_path, sr=16000, duration=30)

    inputs = processor(audio, sampling_rate=16_000, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = detect_model(**inputs)

    # Get the language code with the highest score
    logits = outputs.logits
    predicted_id = torch.argmax(logits, dim=-1).item()
    detected_lang_code = detect_model.config.id2label[predicted_id]

    # Translate the 3-letter ISO code to the full language name
    language_info = pycountry.languages.get(alpha_3=detected_lang_code)

    if language_info:
        print(f"Detected Language: {language_info.name} (Code: {detected_lang_code})")
    else:
        # Fallback just in case pycountry doesn't recognize a very rare code
        print(f"Detected Code: {detected_lang_code} (Name not found in pycountry)")

    return detected_lang_code

# Speech to Text

## SunbirdAI ( For all Ugandan Languages )

In [ ]:
import os
import requests

# SUNBIRD API SETUP
SUNBIRD_API_TOKEN = "Add Your Sunbird AI API Key Here"

# Sunbird Function
def run_sunbirdai(audio_file_path, target_language, verbose=True):
    """
    Uploads audio to Sunbird AI for transcription.
    Languages: lug (Luganda), ach (Acholi), nyn (Runyankole), eng (English)
    """
    if verbose:
        print(f"Routing to Sunbird AI ({target_language})...")

    url = "https://api.sunbird.ai/tasks/stt"
    headers = {"Authorization": SUNBIRD_API_TOKEN}

    # Prepare the file
    try:
        with open(audio_file_path, "rb") as f:

            if audio_file_path.lower().endswith('.wav'):
              mime_type = 'audio/wav'
            else:
              mime_type = 'audio/mpeg' # Default to mp3

            files = {'audio': (audio_file_path, open(audio_file_path, 'rb'), mime_type) }
            data = {"language": target_language, "adapter": target_language}

            if verbose:
                print(f"Uploading {audio_file_path}...")

            # Added a 60-second timeout to prevent the script from freezing overnight!
            response = requests.post(url, headers=headers, files=files, data=data, timeout=60)

        # Check result
        if response.status_code == 200:
            result = response.json()
            transcript = result.get("audio_transcription", "")
            if verbose:
                print("\nSUNBIRD SUCCESS")
                print(transcript)

            # Save to file
            out_filename = f"sunbird_{target_language}_result.txt"
            with open(out_filename, "w", encoding="utf-8") as f:
                f.write(transcript)

            if verbose:
                print(f"Saved to {out_filename}")
            return transcript

        else:
            print(f"Sunbird Error {response.status_code}: {response.text}")
            return None

    except Exception as e:
        print(f"Python Error in Sunbird: {e}")
        return None

## Meta MMS ( For DRC + Ethiopia Languages )

In [ ]:
# LOAD META MMS
print("Loading Meta MMS (1B-All)...")
meta_model_id = "facebook/mms-1b-all"

import torch
import librosa
from transformers import Wav2Vec2ForCTC, AutoProcessor

mms_processor = AutoProcessor.from_pretrained(meta_model_id)
mms_model = Wav2Vec2ForCTC.from_pretrained(meta_model_id)
mms_model.to(device)

Loading Meta MMS (1B-All)...


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [ ]:
# Meta MMS Function
def run_meta_mms(audio_file_path, target_language, verbose=True):
    """
    Runs Meta MMS for specific languages using Adapters.
    target_languages: ISO code (e.g., 'orm', 'lin')
    """
    if verbose:
        print(f"Routing to Meta MMS (Language: {target_language})...")
        print(f"   - Loading audio: {audio_file_path}")
        print(f"Switching adapter to '{target_language}'...")

    try:
        # 1. Load Audio & Resample to 16kHz
        audio_input, sample_rate = librosa.load(audio_file_path, sr=16000)

        # 2. Set the Adapter
        mms_processor.tokenizer.set_target_lang(target_language)
        mms_model.load_adapter(target_language)

        # 3. Prepare Inputs
        inputs = mms_processor(audio_input, sampling_rate=16000, return_tensors="pt")
        inputs = inputs.to(device)

        # 4. Run Inference
        with torch.no_grad():
            outputs = mms_model(**inputs)

        # 5. Decode
        ids = torch.argmax(outputs.logits, dim=-1)[0]
        transcription = mms_processor.decode(ids)

        # 6. Print & Save
        if verbose:
            print("\nMETA MMS TRANSCRIPT")
            print(transcription)

        out_filename = f"mms_{target_language}_result.txt"
        with open(out_filename, "w", encoding="utf-8") as f:
            f.write(transcription)

        if verbose:
            print(f"Saved to '{out_filename}'")
        return transcription

    except Exception as e:
        print(f"Error in Meta MMS: {e}")
        return None

# Mapping Function

In [ ]:
# Decides which model to use based on the detected code

def routing_function_with_lang(audio_file):
    lang_code = detect_language_mms(audio_file)
    transcript = None

    if lang_code in ["orm", "lin", "swh", "swa", "hau", "fra"]:
        print(f"Language is {lang_code} -> Routing to Meta MMS.")
        transcript = run_meta_mms(audio_file, lang_code)

    elif lang_code in ["lug", "nyn"]:
        print(f"Language is {lang_code} -> Routing to SUNBIRD AI.")
        transcript = run_sunbirdai(audio_file, lang_code)

    else:
        print(f"Unexpected language ({lang_code}). Defaulting to Meta MMS.")
        transcript = run_meta_mms(audio_file, lang_code)

    return transcript, lang_code

In [ ]:
def targeted_chunk_router(chunk_path, lang_code, verbose=False):
    """
    Routes tiny audio chunks to the correct STT model based on the PRE-DETECTED language.
    Returns ONLY the transcript string.
    """

    if lang_code in ["orm", "lin", "swh", "swa", "hau"]:
        return run_meta_mms(chunk_path, lang_code, verbose=verbose)

    elif lang_code in ["lug", "nyn"]:
        return run_sunbirdai(chunk_path, lang_code, verbose=verbose)

    else:
        return run_meta_mms(chunk_path, lang_code, verbose=verbose)

# Classification using Llama 3.3

In [ ]:
import json
import time
import json
from groq import Groq

# Add Groq api key
client = Groq(api_key="Add Your Groq API Key Here")

def classify_stream_relevance(transcript_text):
    """
    Sends the 30-second native transcript to Groq/Llama 3 to determine if it is
    political/conflict news or irrelevant (sports, music, weather, ads).
    """

    prompt = f"""
    You are a strict data filter for the Armed Conflict Location & Event Data Project (ACLED).
    Read the following short transcript from an African radio stream.

    STEP 1: Identify the primary topic.
    STEP 2: Apply the strict filtering rules below.

    RELEVANT TOPICS:
    1. Politics, government news, or elections.
    2. Armed conflict, rebel activity, or military operations.
    3. Protests, riots, or state violence.
    4. Violent Civilian Casualties (MUST be caused by weapons, armed actors, or state violence).

    STRICTLY IRRELEVANT TOPICS (AUTO-FAIL):
    - Diseases (e.g., cancer, cholera, mpox, malaria), natural deaths, hospital reports, or natural disasters.
    - Sports, religion, advertisement, betting, weather, music, or standard local health news.

    TRANSCRIPT: "{transcript_text}"

    Respond ONLY with a valid JSON object using this exact schema:
    {{
        "primary_topic": "1-3 words describing the main subject (e.g., Cancer symptoms, Rebel ambush, Soccer match)",
        "is_relevant": true,
        "reason": "A 1-sentence explanation of why it passes or fails the rules based on the primary_topic"
    }}
    (Note: Set is_relevant to false if the primary_topic matches any of the strictly irrelevant categories).
    """

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a strict filtering AI. Only output valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Error during relevance classification: {e}")
        return {"is_relevant": False, "reason": "Error parsing LLM response."}

# Audio processing

In [ ]:
def merge_audio_chunks(chunk_list, final_filename):
    """Takes a list of audio files and stitches them together seamlessly using FFmpeg."""
    print(f"Merging {len(chunk_list)} chunks into final file...")

    # FFmpeg requires a text file listing all the pieces to stitch together
    with open('concat_list.txt', 'w') as f:
        for chunk in chunk_list:
            f.write(f"file '{chunk}'\n")

    # Run FFmpeg to concatenate (stitch) them together
    try:
        subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', 'concat_list.txt', '-c', 'copy', final_filename],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        print(f"Final broadcast saved as: {final_filename}")
    except Exception as e:
        print(f"Failed to merge chunks: {e}")

    # Clean up the temporary chunks from Colab storage to save space
    if os.path.exists('concat_list.txt'):
        os.remove('concat_list.txt')
    for chunk in chunk_list:
        if os.path.exists(chunk):
            os.remove(chunk)

In [ ]:
import time
import os
import requests
import subprocess

def record_fixed_duration(stream_url, final_output, num_chunks=10, chunk_duration=30):
    """
    Blindly records a fixed amount of audio (10 chunks of 30s) using a strict FOR loop.
    Guarantees it will stop and return the merged file, even if some chunks get corrupted!
    """
    total_time = num_chunks * chunk_duration
    print(f"\nConfirmed relevance! Recording the next {total_time} seconds continuously...")

    saved_chunks = []
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        response = requests.get(stream_url, headers=headers, stream=True, timeout=15)
        response.raise_for_status()

        # The Loop will run exactly 10 times.
        for i in range(num_chunks):
            chunk_name = f"raw_chunk_{i}.mp3"
            clean_name = f"clean_chunk_{i}.mp3"
            print(f"-> Capturing segment {i+1}/{num_chunks}...")

            chunk_start = time.time()

            # 1. Try to download the chunk
            try:
                with open(chunk_name, 'wb') as f:
                    for byte_chunk in response.iter_content(chunk_size=8192):
                        if byte_chunk:
                            f.write(byte_chunk)
                        if time.time() - chunk_start >= chunk_duration:
                            break # 30 seconds reached, break inner loop!
            except Exception as stream_e:
                print(f"   [!] Stream hiccup during download: {stream_e}")
                # We don't crash, we just proceed to format whatever audio we managed to grab!

            # 2. ISOLATED FFMPEG BLOCK: Check if file exists and isn't empty
            if os.path.exists(chunk_name) and os.path.getsize(chunk_name) > 0:
                try:
                    subprocess.run(['ffmpeg', '-y', '-i', chunk_name, '-ar', '16000', '-ac', '1', clean_name],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
                    saved_chunks.append(clean_name)
                except subprocess.CalledProcessError as ffmpeg_e:
                    print(f"   [!] FFmpeg crashed on chunk {i+1}. Skipping this chunk.")
                finally:
                    # Clean up the raw file whether FFmpeg succeeded or failed
                    if os.path.exists(chunk_name):
                        os.remove(chunk_name)
            else:
                print(f"   [!] Chunk {i+1} was empty. Skipping.")
                if os.path.exists(chunk_name):
                    os.remove(chunk_name)

    except Exception as e:
        print(f"Master streaming error (Connection lost): {e}")

    # 3. Merge whatever successfully survived!
    if saved_chunks:
        merge_audio_chunks(saved_chunks, final_output)
        print(f"Recording complete. Successfully saved {len(saved_chunks)}/{num_chunks} chunks.")
        return final_output

    print("Failed to record any valid audio.")
    return None

In [ ]:
import subprocess

def force_wav_format(audio_path):
    """Converts any audio file to strict 16kHz mono WAV for PyAnnote stability."""
    print(f"Converting to strict WAV format for PyAnnote...")
    wav_path = audio_path.rsplit('.', 1)[0] + "_clean.wav"
    try:
        subprocess.run([
            'ffmpeg', '-y', '-i', audio_path,
            '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le',
            wav_path
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        return wav_path
    except Exception as e:
        print(f"Failed to convert to WAV: {e}")
        return audio_path

# GLiner NER

In [ ]:
import re
import regex as reg

# Clean
def clean_transcript(text):

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove weird repeated characters (aaaaaaa → aa)
    text = reg.sub(r"(.)\1{3,}", r"\1\1", text)

    # Remove random garbage tokens (numbers mixed with letters)
    text = re.sub(r"\b[a-z]*\d+[a-z]*\b", "", text)

    # Strip
    text = text.strip()

    return text

# Chunk
def chunk_text(text, max_words=250):
    words = text.split()
    chunks = []

    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i + max_words])
        chunks.append(chunk)

    return chunks

In [ ]:
# Translate
from deep_translator import GoogleTranslator

def translate_to_english(text, source_lang = "auto", target="en"):
    try:
        # Google limits text length per request (usually ~5000 chars).
        translator = GoogleTranslator(source=source_lang, target="en")
        translation = translator.translate(text)
        return translation
    except Exception as e:
        return f"Error: {e}"

In [ ]:
from gliner import GLiNER

model_gliner = GLiNER.from_pretrained("urchade/gliner_base")

In [ ]:
labels = [
    "person",
    "Organization",
    "location",
    "date",
    "Event"
]

def run_gliner(text, labels):
    entities = model_gliner.predict_entities(text, labels)
    return entities

import dateparser
from datetime import datetime

def normalize_date(date_text):
    parsed = dateparser.parse(date_text)
    if parsed:
        return parsed.strftime("%Y-%m-%d")
    return None

In [ ]:
import json

def process_transcript(raw_text, source_lang, labels):
    cleaned = clean_transcript(raw_text)
    chunks = chunk_text(cleaned)

    all_entities = []
    translated_chunks = []

    for chunk in chunks:
        # 1. Translate
        translated = translate_to_english(chunk, source_lang)
        translated_chunks.append(translated)

        # 2. NER
        entities = run_gliner(translated, labels)

        # 3. Normalize dates
        for ent in entities:
            if ent["label"] == "date":
                ent["normalized_date"] = normalize_date(ent["text"])

        all_entities.extend(entities)

    # Fixing duplicates
    unique_entities = {}
    for ent in all_entities:
        # Create a unique key using the lowercase text and the label
        key = (ent["text"].lower().strip(), ent["label"])

        # If we haven't seen this entity, OR if this mention has a higher confidence score, keep it
        if key not in unique_entities or ent["score"] > unique_entities[key]["score"]:
            unique_entities[key] = ent

    # Convert the dictionary back to a list
    final_entities = list(unique_entities.values())

    # Sort entities by label for nicer reading
    final_entities = sorted(final_entities, key=lambda x: x["label"])

    return {
        "cleaned_original": cleaned,
        "translated_text": " ".join(translated_chunks), # Join chunks back into one readable paragraph
        "entities": final_entities
    }

def print_report(output_data):
    print("NER EXTRACTION REPORT")

    print("\nTRANSLATED TEXT:")
    # Print the text nicely wrapped
    import textwrap
    print(textwrap.fill(output_data["translated_text"], width=80))

    print("\nUNIQUE ENTITIES EXTRACTED:")
    print(f"{'ENTITY':<30} | {'LABEL':<15} | {'CONFIDENCE'}")
    print("-" * 60)

    for ent in output_data["entities"]:
        name = ent["text"].title()
        label = ent["label"].upper()
        score = f"{ent['score']:.2f}"

        # If it has a normalized date, show it
        if "normalized_date" in ent:
            name = f"{name} ({ent['normalized_date']})"

        print(f"{name:<30} | {label:<15} | {score}")
    print("="*60 + "\n")

# Events Extraction with llama

In [ ]:
# THE HYBRID STRUCTURING FUNCTION
def structure_events_with_llm(translated_text, gliner_entities):

    entity_list_str = "\n".join([f"- {ent['text']} ({ent['label']})" for ent in gliner_entities])

    # Capture today's exact date in the DD-MM-YYYY format
    current_date = datetime.now().strftime("%d-%m-%Y")

    prompt = f"""
    You are an expert military and political intelligence analyst coding data for ACLED.
    Your task is to group extracted entities into distinct conflict or political EVENTS based on the provided text.

    The text below is formatted as a transcript with speaker tags (e.g., [SPEAKER_00]). Use these tags to differentiate between different people talking.

    RULES:
    1. ONLY extract events related to political developments, military action, protests, civil unrest, or government affairs.
    2. STRICTLY IGNORE sports, entertainment, advertisements, religion topics, diseases outbreak and health affairs, call to action, weather, education, and casual banter.
    3. ONLY use the entities provided in the "EXTRACTED ENTITIES" list to fill out the Who, Where, and When fields.
    4. ACTOR FILTERING ("Who" field):
       - NEVER include speaker tags (e.g., [SPEAKER_00], [SPEAKER_01]) in the Who field. Extract the actual people, militias, or organizations involved.
    5. DATE FORMATTING ("When" field):
       - All dates MUST be in DD/MM/YYYY format.
       - If the text mentions a relative timeframe (e.g., "today", "yesterday"), calculate the exact date based on today's extraction date: {current_date}.
       - If NO DATE is mentioned at all for the event, you must use today's date and append " (ED)" to denote Extraction Date. Exactly like this: {current_date} (ED).
    6. GRANULARITY MANDATE: Your 'Notes' MUST be highly detailed. You must explicitly mention any reported numbers (casualties, arrests, crowd sizes), specific weapons or tactics (e.g., tear gas, ambush, airstrike), and specific targets. Do NOT write generic summaries.

    OUTPUT FORMAT:
    Output strictly as a JSON object containing a single key "events" which holds a list of event objects.

    EXACT SCHEMA REQUIRED:
    {{
      "events": [
        {{
          "Event_Name": "Name of the event (e.g., Police clash with protesters)",
          "Who": ["List of Persons or Organizations involved (NO SPEAKER TAGS)"],
          "Where": ["List of Locations involved"],
          "When": "Exact Date in DD/MM/YYYY format, or {current_date} (ED) if no date is mentioned",
          "Notes": "A highly granular, journalistic incident report detailing exactly what happened. MUST include any specific casualties, weapons, tactics, numbers, or impacts mentioned."
        }}
      ]
    }}

    TRANSCRIBED SCRIPT:
    {translated_text}

    EXTRACTED ENTITIES FROM GLiNER:
    {entity_list_str}
    """

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a precise data extraction AI. Only output valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            seed=42,
            response_format={"type": "json_object"}
        )

        cleaned_response = response.choices[0].message.content.strip()
        if cleaned_response.startswith("```json"):
            cleaned_response = cleaned_response[7:]
        if cleaned_response.endswith("```"):
            cleaned_response = cleaned_response[:-3]

        return json.loads(cleaned_response.strip())

    except Exception as e:
        print(f"Error during LLM structuring: {e}")
        return None

# Print report
def print_final_acled_report(structured_data):

    events = structured_data.get("events", [])

    if not events:
        print("No events structured.")
        return

    for i, event in enumerate(events, 1):
        print(f"\nEVENT [{i}]: {event.get('Event_Name', 'Unknown').upper()}")
        print("-" * 70)
        print(f"Notes   : {event.get('Notes', 'N/A')}")
        print(f"Who     : {', '.join(event.get('Who', ['None']))}")
        print(f"Where   : {', '.join(event.get('Where', ['None']))}")
        print(f"When    : {event.get('When', 'None')}")

In [ ]:
import os
import shutil
import json
from google.colab import drive

# 1. Ensure Drive is mounted and create the storage folder
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/ACLED_Radio_Extractions"
os.makedirs(SAVE_DIR, exist_ok=True)

def get_country_prefix(lang_code):
    """Maps language codes to the target conflict zones for file naming."""
    mapping = {
        'lug': 'uganda', 'nyn': 'uganda', 'ach': 'uganda',
        'swh': 'uganda', 'swa': 'uganda',
        'orm': 'ethiopia', 'amh': 'ethiopia', 'hau': 'ethiopia',
        'lin': 'drc', 'fra': 'drc'
    }
    return mapping.get(lang_code, 'unclassified')

def get_next_index(prefix):
    """Scans the Drive folder to find the next available file number."""
    existing_files = os.listdir(SAVE_DIR)
    indices = []
    for f in existing_files:
        if f.startswith(prefix):
            try:
                # Extracts the number 'i' from format 'prefix_type_i.ext'
                idx = int(f.split('_')[-1].split('.')[0])
                indices.append(idx)
            except ValueError:
                pass
    return max(indices) + 1 if indices else 1

# Full Pipeline

In [ ]:
import time

LOG_FILE = "/content/pipeline_history.txt"

# Clear the log file when you start a fresh session
with open(LOG_FILE, "w", encoding="utf-8") as f:
    f.write(f"=== Pipeline Started at {time.ctime()} ===\n")

def write_log(message):
    """Writes the message to a file permanently."""
    timestamp = time.strftime('%H:%M:%S')
    log_entry = f"[{timestamp}] {message}"

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(log_entry + "\n")

In [ ]:
def format_event_report(audio_file, stream_url, raw_transcript, translated_text, gliner_entities, structured_data):
    """
    Converts all pipeline data into a comprehensive, human-readable report format.
    """
    report_lines = []

    # [DEBUG INFO] To help see if there are errors in the returned data
    print(f"\n[DEBUG] Raw structured_data: {structured_data}")

    # 1. Header & Meta Data
    report_lines.append("ACLED EVENTS REPORT\n")
    report_lines.append(f"Audio File: {audio_file}")
    report_lines.append(f"Radio Stream: {stream_url}\n")

    # 2. Raw Transcript
    report_lines.append("RAW TRANSCRIPT")
    report_lines.append(f"{raw_transcript}\n")

    # 3. Translated Text
    report_lines.append("TRANSLATED TEXT")
    report_lines.append(f"{translated_text}\n")

    # 4. GLiNER Entities (Formatted as a table)
    report_lines.append("GLINER ENTITIES")
    if gliner_entities:
        # Table Header
        report_lines.append(f"{'ENTITY TEXT':<40} | {'LABEL'}")
        report_lines.append("-" * 60)
        # Table Rows
        for ent in gliner_entities:
            report_lines.append(f"{ent['text']:<40} | {ent['label']}")
    else:
        report_lines.append("No entities extracted.")
    report_lines.append("\n")

    # 5. Structured Events
    report_lines.append("STRUCTURED EVENTS")

    # SAFE EXTRACTION: Ensures it reads 'events' correctly
    events = []
    if structured_data and isinstance(structured_data, dict):
        events = structured_data.get("events", [])

    if not events:
        report_lines.append("\nNo relevant political or conflict events detected in this broadcast.")
        report_lines.append(f"\n[DEBUG INFO] The LLM returned: {structured_data}")
    else:
        for i, event in enumerate(events, 1):
            report_lines.append(f"\nEVENT [{i}]: {event.get('Event_Name', 'Unknown').upper()}")
            report_lines.append("-" * 70)

            report_lines.append(f"Notes     : {event.get('Notes', 'N/A')}")

            who = event.get('Who', ['None'])
            where = event.get('Where', ['None'])
            who_str = ", ".join(who) if isinstance(who, list) else str(who)
            where_str = ", ".join(where) if isinstance(where, list) else str(where)

            report_lines.append(f"Who       : {who_str}")
            report_lines.append(f"Where     : {where_str}")
            report_lines.append(f"When      : {event.get('When', 'None')}")

    return "\n".join(report_lines)

In [ ]:
import os
import shutil
import time
import json
import gc
import torch
from IPython.display import clear_output

# 1. LOGGER SETUP
# Point the log file directly to Google Drive folder!
LOG_FILE = os.path.join(SAVE_DIR, "pipeline_history.txt")

def write_log(message):
    """Writes the message to a file permanently."""
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S')
    log_entry = f"[{timestamp}] {message}"

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(log_entry + "\n")

# Log the start of a new session (Using 'a' so we don't delete yesterday's logs!)
with open(LOG_FILE, "a", encoding="utf-8") as f:
    f.write(f"\n{'='*40}\n")
    f.write(f"=== NEW SESSION STARTED: {time.ctime()} ===\n")
    f.write(f"{'='*40}\n")


# 2. THE MAIN LOOP
def continuous_acled_pipeline(stream_url):

    write_log(f"Started End-to-End Monitor: {stream_url}")
    write_log(f"Safely syncing extractions to: {SAVE_DIR}")

    chunks_processed = 0
    events_found = 0

    while True:
        chunks_processed += 1

        # 1. WIPE THE SCREEN
        clear_output(wait=True)

        #2. PRINT THE STATIC DASHBOARD
        print("="*60)
        print("ACLED RADIO MONITORING PIPELINE")
        print(f"Stream: {stream_url}")
        print(f"Chunks Processed: {chunks_processed}")
        print(f"Events Detected: {events_found}")
        print(f"Log File: {LOG_FILE}")
        print("="*60)

        #3. BEGIN PROCESSING
        print("▶ Currently: Listening for Human Speech...")
        write_log(f"Processing chunk #{chunks_processed}")

        audio_chunk_path = wait_for_speech_and_sample(stream_url, duration=30)

        if not audio_chunk_path:
            write_log("No speech detected. Sleeping for 10 seconds.")
            time.sleep(3)
            continue

        print("▶ Currently: Initial Transcription & Language Detection...")
        write_log("Speech detected. Running initial transcription...")
        transcript, lang_code = routing_function_with_lang(audio_chunk_path)

        if not transcript or not transcript.strip():
            write_log("Transcript empty. Skipping.")
            continue

        # RELEVANCE CHECK
        print(" Currently: Checking Relevance (Llama 3.3)...")
        write_log(f"Checking relevance for {lang_code} transcript...")

        # 1. Catch the dictionary returned by the function
        classification = classify_stream_relevance(transcript)

        # 2. Extract the values safely
        is_relevant = classification.get("is_relevant", False)
        reason = classification.get("reason", "No reason provided")

        if not is_relevant:
            print(f"Irrelevant ({reason}). Sleeping for 2.5 minute...")
            write_log(f"Irrelevant ({reason}). Sleeping for 2.5 minute.")
            time.sleep(150)
            continue

        # Event found, Update dashboard stats
        events_found += 1

        print(f"\nRELEVANT EVENT DETECTED! ({reason})")
        print("▶ Currently: Recording Full Broadcast...")
        write_log(f"RELEVANT EVENT DETECTED! Reason: {reason}. Recording full broadcast...")

        # Record locally first
        temp_audio_file = f"temp_broadcast_{int(time.time())}.mp3"
        final_audio_path = record_fixed_duration(stream_url, final_output=temp_audio_file)

        if not final_audio_path:
            write_log("Failed to record broadcast.")
            continue

        print("▶ Currently: Formatting Audio (Ensuring safe WAV format)...")
        write_log("Formatting audio to strict WAV format...")
        safe_wav_path = force_wav_format(final_audio_path)

        if not safe_wav_path:
            write_log("Audio formatting failed. Skipping.")
            continue

        print(" Currently: Mapping Speakers (Diarization)...")
        write_log("Running Pyannote Speaker Diarization...")
        speaker_segments = run_diarization(safe_wav_path)

        if speaker_segments:
            print("Currently: Full NLP Pipeline (Stitched Transcription -> Translation -> GLiNER -> Structuring)...")
            write_log("Running full NLP Pipeline...")

            full_transcript = transcribe_and_stitch_diarization(safe_wav_path, speaker_segments, lang_code)

            if full_transcript and full_transcript.strip():

                write_log("Running GLiNER and Event Structuring...")
                labels = ["person", "Organization", "location", "date", "Event"]
                ner_results = process_transcript(full_transcript, source_lang="auto", labels=labels)

                structured_events = structure_events_with_llm(
                    ner_results["translated_text"],
                    ner_results["entities"]
                )

                print("Currently: Saving Data to Google Drive...")
                write_log("Saving structured data to Google Drive...")
                prefix = get_country_prefix(lang_code)
                idx = get_next_index(prefix)

                drive_audio = os.path.join(SAVE_DIR, f"{prefix}_audio_{idx}.wav")
                drive_transcript = os.path.join(SAVE_DIR, f"{prefix}_transcript_{idx}.txt")
                drive_event = os.path.join(SAVE_DIR, f"{prefix}_event_{idx}.txt")

                # Move files safely
                shutil.move(safe_wav_path, drive_audio)

                with open(drive_transcript, "w", encoding="utf-8") as f:
                    f.write(full_transcript)

                formatted_report = format_event_report(
                    audio_file=os.path.basename(drive_audio),
                    stream_url=stream_url,
                    raw_transcript=full_transcript,
                    translated_text=ner_results["translated_text"],
                    gliner_entities=ner_results["entities"],
                    structured_data=structured_events
                )

                with open(drive_event, "w", encoding="utf-8") as f:
                    f.write(formatted_report)

                # Show the report on screen, then pause so you can read it before the screen wipes!
                print("\n" + formatted_report + "\n")
                write_log(f"SUCCESS: Event {idx} successfully saved to Drive.")
                time.sleep(5)

            else:
                write_log("Full stitched transcription failed.")
        else:
            write_log("Diarization failed.")

        # DISK LEAK CLEANUP
        if os.path.exists(final_audio_path):
            os.remove(final_audio_path)
            write_log("Cleaned up temporary MP3.")

        # THE MEMORY FLUSH
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        write_log("Memory flushed. Returning to listen mode.")

In [ ]:
# test
stream_url = 'http://rs1.radiostreamer.com:8000/;'
continuous_acled_pipeline(stream_url)

ACLED RADIO MONITORING PIPELINE
Stream: http://rs1.radiostreamer.com:8000/;
Chunks Processed: 31
Events Detected: 2
Log File: /content/drive/MyDrive/ACLED_Radio_Extractions/pipeline_history.txt
▶ Currently: Listening for Human Speech...
 Attempting to record from: http://rs1.radiostreamer.com:8000/;
Audio saved to: sample.mp3
Analyzing audio for human speech...
Speech detected: 96.2%
Decision: KEEP (Human speech detected!)
▶ Currently: Initial Transcription & Language Detection...
Listening to clean_sample.mp3...
Detected Language: Lingala (Code: lin)
Language is lin -> Routing to Meta MMS.
Routing to Meta MMS (Language: lin)...
   - Loading audio: clean_sample.mp3
Switching adapter to 'lin'...

META MMS TRANSCRIPT
liie l balesi eidim e u puee penir shemoie u sa ee pasaauadi ei mariidelieie le pauet ev ate sasie aprueelaeiaiiuiaeilodola depuis kinchasa ell vise ui denonce lattitude de certains hommes a manièra leur manièra
Saved to 'mms_lin_result.txt'
▶ Currently: Checking Relevance (